# Parking V1 — matched 0–6 m GPU pilot
**PARKING-V1-MATCHED-NEAR-PILOT-2026-10-30-r1**

Purpose: compare ordinary 2–6 m continuation with six-cell 0–6 m supervision. Both start from unchanged V0 (actual-camera epoch 4), train only its existing 1,553-parameter camera adapter, and use identical draws and 1,920 updates each. No new architecture, KD, held-out scoring or phone update.

Each arm has **five blocks of 384 updates**, not five full dataset epochs. Stop after that budget; return results for review. V0 can remain the best endpoint.

## How to run
1. On a **CPU runtime**, run sections 1–3 to verify the prepared archive set. No full nuScenes download is needed.
2. Select an **A100 or L4 GPU**. If Colab reconnects/reset local disk, rerun sections 1–3, then 4–7.
3. Section 5 caches frozen features once on GPU. Both arms reuse this local cache.
4. Section 6 verifies unchanged V0 predictions against the completed CPU result and checks both arms' gradients **before training**.

Inputs: reviewed preparation summary ZIP; ten `train_partNN.zip` archives (~775 MB total); existing dev-only `parking_v0_dev_inputs.zip`; completed `v0_complete.zip`. All active I/O/logs stay under `/content`. Do not use the full sensor dataset or the old training package.

**Resume:** same VM: rerun the failed section with the same RUN_ID. A partial training block restarts from the last completed block; completed blocks are not repeated. After VM reset: set RESUME_ARCHIVE to the last printed progress ZIP (or upload it), rebuild the verified local inputs/features in the same GPU/runtime, then rerun. Changed code, GPU or package identity fails closed. No automatic remount/retry for Drive I/O errors.



## 1. Paths, local logging and main checkout
Edit source paths only if your archives are elsewhere. Optional uploads in `/content` take priority over Drive. Preserve completed V0 and training-input folders.


In [ ]:
from pathlib import Path
from collections import deque
import datetime, json, os, re, shlex, subprocess, sys
REVISION = 'PARKING-V1-MATCHED-NEAR-PILOT-2026-10-30-r1'
RUN_ID = 'parking_v1_matched_adapter_0_6_r1'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID): raise ValueError('Invalid RUN_ID')
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive')
MOBILE = Path('/content/mobile_adas3d')
PREPARATION = BASE/'mobile_adas3d_outputs/parking_v1_training_inputs/parking_v1_six_cell_training_inputs_r1'
SUMMARY = Path('/content/parking_v1_training_inputs_results.zip')
if not SUMMARY.is_file(): SUMMARY = PREPARATION/'parking_v1_training_inputs_results.zip'
PARTS = Path('/content/train_parts')
if not PARTS.is_dir(): PARTS = PREPARATION/'train_parts'
DEV_CAPSULE = Path('/content/parking_v0_dev_inputs.zip')
if not DEV_CAPSULE.is_file(): DEV_CAPSULE = BASE/'mobile_adas3d_inputs/parking_v0_dev/epoch4_frozen512_r1/parking_v0_dev_inputs.zip'
V0_RESULT = Path('/content/v0_complete.zip')
if not V0_RESULT.is_file(): V0_RESULT = BASE/'mobile_adas3d_outputs/parking_v0_dev/parking_v0_actual_camera_epoch4_local_r2/portable_checkpoints/v0_complete.zip'
INPUTS = Path('/content')/('parking_v1_pilot_inputs_'+RUN_ID)
CACHE = Path('/content')/('parking_v1_pilot_features_'+RUN_ID)
OUTPUT = Path('/content')/('parking_v1_pilot_run_'+RUN_ID)
SOURCE = Path('/content/Urban_Parking_V1_Source')
WEIGHTS = Path('/content/parking_v1_weights')
DEPS = Path('/content')/('parking_v1_pilot_deps_'+RUN_ID)
REMOTE_PROGRESS = BASE/'mobile_adas3d_outputs/parking_v1_matched_pilot'/RUN_ID/'portable_progress' if USE_DRIVE else None
# After a VM reset, set this to the latest completed-block archive printed in section 6.
# It may be an uploaded /content ZIP or one Drive ZIP. Never use v0_complete.zip here.
RESUME_ARCHIVE = None
PYTHON = sys.executable
CPU_STAGE_PASSED = False
CUDA_READY = False

def run(command, name, cwd=None, task_env=True, gpu=False):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'; path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env.update(YOLO_AUTOINSTALL='false', CUBLAS_WORKSPACE_CONFIG=':4096:8',
               OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
    if not gpu: env['CUDA_VISIBLE_DEVICES']=''
    if task_env: env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(MOBILE)])
    print('+',shlex.join(command),'\nLOCAL log:',path,flush=True)
    tail=deque(maxlen=30)
    with path.open('w') as stream:
        process=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);stream.write(line);stream.flush();tail.append(line.rstrip())
            status=process.wait()
        except BaseException:
            process.terminate();process.wait();raise
        finally: process.stdout.close()
        stream.write(f'\nEXIT CODE: {status}\n')
    if status: raise RuntimeError(f'Exit {status}; preserve local files and inspect {path}\n'+'\n'.join(tail))
    return path

url='/'.join(['https:','','github.com','Ali-RT','mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone','--branch','main',url,MOBILE],'clone',task_env=False)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch!='main': raise RuntimeError(f'Use a clean main checkout; preserve branch {branch!r}')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE,text=True)
if dirty.strip(): raise RuntimeError('Tracked changes exist. Preserve them; this notebook will not reset or overwrite them.')
run(['git','pull','--ff-only'],'update_main',MOBILE,task_env=False)
print(REVISION,'\nInput summary:',SUMMARY,'\nTrain archives:',PARTS,'\nV0 result:',V0_RESULT)



## 2. Pinned task packages and tests (CPU)
This installs preprocessing/model-loader packages into a task-local directory. It does not replace Colab's PyTorch/CUDA or modify the original V0 environment. GPU-vs-CPU V0 parity is a mandatory later gate.


In [ ]:
packages=['ultralytics==8.4.61','ultralytics-thop==2.0.18','numpy==2.1.3','pillow==11.3.0','opencv-python-headless==5.0.0.93','scipy==1.16.3']
run([PYTHON,'-m','pip','install','--quiet','--no-deps','--upgrade','--target',DEPS,*packages],'task_packages',MOBILE,task_env=False)
SCRIPT = MOBILE/'scripts/train_parking_v1_near.py'
DATA_SCRIPT = MOBILE/'scripts/parking_v1_pilot_data.py'
for path in [SCRIPT,DATA_SCRIPT]:
    if not path.is_file(): raise FileNotFoundError(f'{path} missing. Reload the published main notebook.')
run([PYTHON,'-m','unittest','discover','-s','tests','-p','test_parking_v1_matched_pilot.py','-v'],'pilot_tests',MOBILE)



## 3. Verify and stage reusable archives locally (CPU)
One sequential copy per archive; complete local copies are reused without reopening Drive sources. Every RGB, near GT and broad GT body is reverified. Only training bodies and the 512 development images are opened. No coverage-chunk scan, sensor-archive scan or GPU work.

If a source is missing, upload that existing ZIP or correct its path in section 1. If Drive reports I/O failure, stop repeated retries; use uploaded archives or wait for recovery.


In [ ]:
run([PYTHON,'-u',DATA_SCRIPT,'--inputs',INPUTS,'--summary',SUMMARY,'--parts',PARTS,
     '--dev-capsule',DEV_CAPSULE,'--v0-result',V0_RESULT],'cpu_stage',MOBILE)
prepared=json.loads((INPUTS/'prepared_inputs.json').read_text())
assert prepared['complete'] and prepared['train_images']==4096 and prepared['dev_images']==512
assert prepared['heldout_bodies_opened']==0 and prepared['train_benchmark_camera_keyframe_log_overlap']==0
CPU_STAGE_PASSED=True
print('CPU staging passed. Select a GPU for sections 4–6. If reset, rerun 1–3 with the same paths/RUN_ID.')



## 4. GPU allocation and local capacity
No CUDA extension build is needed. At least 12 GiB of free local disk is required for the feature cache and outputs. A changed GPU/runtime cannot silently resume the old run.


In [ ]:
if not CPU_STAGE_PASSED: raise RuntimeError('Run sections 1–3 first.')
import shutil
if shutil.disk_usage('/content').free < 12*1024**3: raise RuntimeError('Need at least 12 GiB of free local disk.')
run([PYTHON,'-c',"import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.__version__, torch.cuda.get_device_name(0)); print(torch.ones(8,device='cuda').sum().item())"],'cuda_preflight',MOBILE,gpu=True)
CUDA_READY=True
def stage(name):
    if not CUDA_READY or not CPU_STAGE_PASSED: raise RuntimeError('Run sections 1–4 first.')
    command=[PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--inputs',INPUTS,'--cache',CACHE,
             '--source',SOURCE,'--weights',WEIGHTS,'--output',OUTPUT,'--device','cuda']
    if REMOTE_PROGRESS is not None: command += ['--publish',REMOTE_PROGRESS]
    if RESUME_ARCHIVE is not None: command += ['--resume-archive',RESUME_ARCHIVE]
    run(command,'v1_'+name,MOBILE,gpu=True)



## 5. Cache frozen image features once (GPU)
This is GPU inference, not training. It verifies the pretrained body, loads unchanged V0 and creates the local feature cache shared by both arms. Completed feature ZIPs are verified and reused. Do not launch another copy of the notebook in parallel.


In [ ]:
stage('cache')
cache_receipt=json.loads((CACHE/'complete.json').read_text())
assert cache_receipt['train_images']==4096 and cache_receipt['dev_images']==512
print('Frozen feature cache ready. No optimizer updates yet.')



## 6. Mandatory V0 replay, gradient smoke, then matched training (GPU)
V0 must reproduce the reviewed CPU point predictions within the declared FP32 tolerance. Both losses must have finite gradients while all frozen state remains exact. Only then are the two 1,920-update arms run.

Each completed block saves one progress ZIP locally and, when available, to Drive. On a cell stop, rerun this section. Partial blocks restart from the previous completed block. No automatic extra blocks. Watch for a failed Drive-save warning and download the printed local progress ZIP before closing the VM.


In [ ]:
stage('smoke')
smoke=json.loads((OUTPUT/'runtime_smoke.json').read_text())
assert smoke['complete'] and smoke['v0_replay_passed'] and smoke['optimizer_steps']==0
stage('train')
result=json.loads((OUTPUT/'v1_pilot_results.json').read_text())
assert result['complete'] and result['updates_per_arm']==1920
assert result['matched_sampling_streams'] and result['frozen_body_base_depth_bn_unchanged']
assert result['heldout_bodies_opened']==0 and not result['heldout_evaluated'] and not result['phone_app_changed']
for arm,value in result['arms'].items():
    print(arm, 'selected block',value['selected_block'],
          'selected 2–6m MAE',value['selected_dev']['conditional_equal_four_cell_mae_m'],
          'fixed block-5 MAE',value['fixed_block_5_dev']['conditional_equal_four_cell_mae_m'])



## 7. Download for review
Return this archive. It includes both arms, all completed block checkpoints and development predictions, source/runtime identities, V0 replay and paired comparisons. No training RGB, held-out bodies or feature cache is included.

Report both the selected endpoint and fixed block 5. Check 0–2 m signed bias/support and every 2–6 m class/range cell; a lower overall number does not guarantee every cell improved. No ±10 cm pass/fail or deployment claim.


In [ ]:
RESULTS=OUTPUT/'portable/parking_v1_matched_pilot_results.zip'
if not RESULTS.is_file(): raise FileNotFoundError('Pilot incomplete. Resume section 6; do not create a new RUN_ID to discard progress.')
print('Return this file:',RESULTS,'\nBytes:',RESULTS.stat().st_size)
from google.colab import files
files.download(str(RESULTS))

